# Look inside a real Qwen3 forward pass: shapes, the next-token distribution,

Part of the free interactive course **Qwen Atlas** — chapter: [hf](https://happy-bamboo-w7dk.here.now/#/learn/hf).

```text
hidden states of every block and one attention map.

Run:  python 13_hf_inspect.py      (needs: torch, transformers; downloads Qwen3-0.6B, about 1.5 GB)
```

The output under the code cell was recorded on 2026-10-23 with Python 3.11.9, numpy 2.4.6, torch 2.14.1+cpu and transformers 5.20.2 on a CPU. Run the cell to reproduce it.


In [ ]:
%pip install -q "transformers>=4.56"

In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = os.environ.get("QWEN_MODEL_DIR") or "Qwen/Qwen3-0.6B"
REV = None if os.path.isdir(MODEL) else "c1899de289a04d12100db370d81485cdf75e47ca"
tok = AutoTokenizer.from_pretrained(MODEL, revision=REV)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, dtype=torch.float32, attn_implementation="eager")
model.eval()

print(model.model.layers[0])                     # one decoder block: attention + MLP + two RMSNorms

ids = tok("The capital of France is", return_tensors="pt").input_ids
with torch.no_grad():
    out = model(ids, output_hidden_states=True, output_attentions=True)

print("\ninput ids      :", ids.tolist()[0])
print("logits shape   :", tuple(out.logits.shape), "= [batch, positions, vocabulary rows]")
p = out.logits[0, -1].softmax(-1)
top = p.topk(5)
for prob, i in zip(top.values, top.indices):
    print(f"   next token {tok.decode([int(i)])!r:<10} p = {prob:.3f}")

hs = out.hidden_states                           # embedding, blocks 0..26, then the final norm output
print("\nhidden states  :", len(hs), "tensors of shape", tuple(hs[0].shape))
names = {0: "embedding", 1: "block 0", 14: "block 13", 27: "block 26", 28: "final norm"}
for l, name in names.items():
    print(f"   after {name:<10} RMS of last position = {hs[l][0, -1].pow(2).mean().sqrt():.3f}")

att = out.attentions[13][0, 0]                   # block 13, head 0: [query, key]
print("\nattention block 13 head 0 (rows sum to 1, upper triangle is 0 = causal):")
print(att.numpy().round(3))

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)

input ids      : [785, 6722, 315, 9625, 374]
logits shape   : (1, 5, 151936) = [batch, positions, vocabulary rows]
   next token ' Paris'   p = 0.657
   next token ' located' p = 0.02